# Understanding Learning Curves

In [1]:
from datasets import load_dataset

# Dataset id from huggingface.co/dataset
dataset_id = "burtenshaw/PleIAs_common_corpus_code_classification"

# Load raw dataset
dataset = load_dataset(dataset_id)

In [2]:
print(len(dataset["train"]))
print(dataset["train"][0])

127723
{'text': '/*\n * Copyright (c) 2000 Kungliga Tekniska Högskolan\n * (Royal Institute of Technology, Stockholm, Sweden).\n * All rights reserved.\n *\n * Redistribution and use in source and binary forms, with or without\n * modification, are permitted provided that the following conditions\n * are met:\n *\n * 1. Redistributions of source code must retain the above copyright\n *    notice, this list of conditions and the following disclaimer.\n *\n * 2. Redistributions in binary form must reproduce the above copyright\n *    notice, this list of conditions and the following disclaimer in the\n *    documentation and/or other materials provided with the distribution.\n *\n * 3. Neither the name of the Institute nor the names of its contributors\n *    may be used to endorse or promote products derived from this software\n *    without specific prior written permission.\n *\n * THIS SOFTWARE IS PROVIDED BY THE INSTITUTE AND CONTRIBUTORS ``AS IS\'\' AND\n * ANY EXPRESS OR IMPLIED W

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Model id to load the tokenizer'
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Prepare model labels - useful for inference
train_labels = sorted(set(dataset["train"]["labels"]))
label2id = {label: i for i, label in enumerate(train_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label2id)
dataset["test"] = dataset["test"].filter(lambda x: x["labels"] in label2id)

if "validation" in dataset:
    dataset["validation"] = dataset["validation"].filter(lambda x: x["labels"] in label2id)

# helper function
def preprocess(batch):
    # tokenize
    tokenized = tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")
    # encode labels
    tokenized["labels"] = [label2id[label] for label in batch["labels"]]
    
    return tokenized


tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=["text"])

print(tokenized_dataset["train"].features.keys())
# dict_keys(['labels', 'input_ids', 'attention_mask'])

dict_keys(['labels', 'input_ids', 'attention_mask'])


In [4]:
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

# # Model id to load the tokenizer
# model_id = "answerdotai/ModernBERT-base"

# # Load Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# def tokenize_function(example):
#     return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


# tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["sentence1", "sentence2", "idx"])
# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# num_labels = len(set(tokenized_dataset["train"]["label"]))

In [6]:
# # Download the model from huggingface.co/models
# model = AutoModelForSequenceClassification.from_pretrained(
#     model_id, num_labels=num_labels)

In [7]:
import numpy as np
from sklearn.metrics import f1_score

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}


In [ ]:
from huggingface_hub import get_token
from transformers import Trainer, TrainingArguments

# Define training args
training_args = TrainingArguments(
    output_dir= "ModernBERT-code-classifier",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True, # bfloat16 training
    optim="adamw_torch_fused", # improved optimizer
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # push to hub parameters
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=get_token(),
    report_to="trackio",
    project="modernbert-code",
    trackio_space_id="ModernBert-code-classifier",
)



In [9]:
training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False

# Overfitting

In [10]:
limited_dataset = tokenized_dataset["train"].select(range(100))

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

* Trackio project initialized: modernbert-code
* Trackio metrics will be synced to Hugging Face Bucket: https://huggingface.co/buckets/hwting/ModernBert-code-classifier-bucket
* Found existing space: https://huggingface.co/spaces/hwting/ModernBert-code-classifier


* Created new run: hwting-1776059477


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 15.61 GiB of which 1.83 GiB is free. Including non-PyTorch memory, this process has 13.76 GiB memory in use. Of the allocated memory 9.87 GiB is allocated by PyTorch, and 3.65 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# clear memory

import torch
torch.cuda.empty_cache()

# del trainer
# del model
# del limited_dataset

# Underfitting

In [ ]:
# define a low learning rate
training_args.learning_rate = 1e-7

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

* Created new run: hwting-1775790874


Epoch,Training Loss,Validation Loss,F1
1,No log,0.874459,0.727189
2,No log,0.871110,0.728702
3,No log,0.869097,0.726212
4,No log,0.867297,0.726352
5,No log,0.865905,0.726352


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

* Run finished. Uploading logs to Trackio Space (please wait...)


TrainOutput(global_step=35, training_loss=0.1725019727434431, metrics={'train_runtime': 167.1579, 'train_samples_per_second': 2.991, 'train_steps_per_second': 0.209, 'total_flos': 17773316948940.0, 'train_loss': 0.1725019727434431, 'epoch': 5.0})

In [ ]:
# clear memory

import torch
torch.cuda.empty_cache()

# del trainer
# del model

# Just right! 🥣

In [ ]:
# define a valid learning rate
training_args.learning_rate = 5e-5

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

* Created new run: hwting-1775791473


Epoch,Training Loss,Validation Loss,F1
1,No log,1.055747,0.708529
2,No log,0.990148,0.697835
3,No log,1.238521,0.712926
4,No log,1.158786,0.718327
5,No log,1.161202,0.719757


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

* Run finished. Uploading logs to Trackio Space (please wait...)


TrainOutput(global_step=35, training_loss=1.736484854561942, metrics={'train_runtime': 164.2516, 'train_samples_per_second': 3.044, 'train_steps_per_second': 0.213, 'total_flos': 17773316948940.0, 'train_loss': 1.736484854561942, 'epoch': 5.0})

# Inference

In [ ]:
from transformers import pipeline

# load model from huggingface.co/models using our repository id
classifier = pipeline(
    task="text-classification",
    model="hwting/ModernBERT-code-classifier",
    device=0,
)

sample = """def add_numbers(a, b):
    return a + b"""

classifier(sample)
